# LeetCode 3989 — Maximum Consistent Columns in a Grid bằng Python

- Link: https://leetcode.com/problems/maximum-consistent-columns-in-a-grid/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động (DP), Dãy con dài nhất kiểu LIS, Ma trận, Bitset

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxConsistentColumns(self, grid: List[List[int]], limit: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho ma trận `grid` kích thước `m × n` và số `limit`. Ta được **xoá một số cột** (giữ lại ít nhất một cột, giữ nguyên thứ tự). Ma trận còn lại gọi là **nhất quán** nếu với **mỗi hàng**, hai cột **liền kề** nhau trong phần còn lại đều chênh nhau không quá `limit`. Tìm số cột nhiều nhất có thể giữ.

Bài này là "LIS biến thể": thay vì "tăng dần", điều kiện giữa hai phần tử liền kề là **tương thích** (compatible): tất cả `m` hàng đều thoả `|a − b| ≤ limit`. Điều khó nhận ra là điều kiện chỉ áp dụng giữa các cột **liền kề trong dãy được giữ**, **không** áp dụng cho mọi cặp cột được giữ. Nhờ vậy, đáp án là **đường đi dài nhất** trong DAG các cột (cạnh `i → j` nếu `i < j` và hai cột tương thích): `dp[j] = 1 + max dp[i]`.

Sau notebook này bạn sẽ:

1. Phát biểu chính xác "nhất quán" và phân biệt điều kiện **liền kề** với điều kiện **mọi cặp**.
2. Viết brute force liệt kê mọi tập con cột (với `n` nhỏ) làm trọng tài.
3. Viết DP `O(n² · m)` kiểu LIS, rồi tách bước **tính ma trận tương thích** khỏi bước DP.
4. Tăng tốc bằng `numpy` và bằng **bitset** + sắp xếp + chặt nhị phân (`bisect`) cho mỗi hàng: tính tương thích trong `O(m · n log n)` phép bit.
5. Đối chiếu các cách bằng test ngẫu nhiên có seed, đo thời gian ở `m = n = 250`.

## 2. Đề bài & ví dụ

**Phát biểu lại.**

- Cho `grid` (`m` hàng, `n` cột) và `limit ≥ 0`.
- Chọn một **tập con khác rỗng** các chỉ số cột `c_1 < c_2 < ... < c_k` (thứ tự tương đối giữ nguyên).
- Hợp lệ (**nhất quán**) nếu với mọi hàng `r` và mọi `t = 1..k-1`: `|grid[r][c_{t+1}] − grid[r][c_t]| ≤ limit`.
- Trả về `k` lớn nhất.

**Ví dụ 1:** `grid = [[-2,0,3]]`, `limit = 2` → `2`: giữ cột 0 và 1 (`|0−(−2)| = 2 ≤ 2`).

**Ví dụ 2:** `grid = [[1,-1,1],[2,2,2]]`, `limit = 1` → `2`: bỏ cột 1, giữ cột 0 và 2 (hàng 0: `|1−1| = 0`; hàng 1: `|2−2| = 0`).

**Ví dụ 3:** `grid = [[-5,5]]`, `limit = 9` → `1`: hai cột chênh `10 > 9`.

**Ràng buộc**

- `1 <= m, n <= 250`
- `-10^5 <= grid[i][j] <= 10^5`
- `0 <= limit <= 10^5`

In [1]:
from typing import List

EXAMPLES = [
    ([[-2, 0, 3]], 2, 2),
    ([[1, -1, 1], [2, 2, 2]], 1, 2),
    ([[-5, 5]], 9, 1),
]
for grid, limit, expected in EXAMPLES:
    print(f"grid={grid}, limit={limit} -> kỳ vọng {expected}")

grid=[[-2, 0, 3]], limit=2 -> kỳ vọng 2
grid=[[1, -1, 1], [2, 2, 2]], limit=1 -> kỳ vọng 2
grid=[[-5, 5]], limit=9 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Cột của ma trận: `zip(*grid)`

`grid` là list các hàng. Để lấy **cột**, ta "chuyển vị": `list(zip(*grid))` cho list các tuple, mỗi tuple là một cột. `*grid` giải gói các hàng thành các đối số của `zip`.

In [2]:
grid = [[1, -1, 1], [2, 2, 2]]
cols = list(zip(*grid))
print(cols)                 # [(1, 2), (-1, 2), (1, 2)]
print(cols[1][0], grid[0][1])    # cùng một giá trị: hàng 0 cột 1

[(1, 2), (-1, 2), (1, 2)]
-1 -1


### 3.2. `all(...)`, `abs` và cặp cột tương thích

Hai cột `a`, `b` **tương thích** nếu mọi hàng thoả `|a_r − b_r| ≤ limit`. Trong Python: `all(abs(x - y) <= limit for x, y in zip(a, b))`. `all` dừng ngay ở hàng đầu tiên vi phạm (đoản mạch), nên thường nhanh hơn `max(...)`.

In [3]:
def compatible(a, b, limit):
    return all(abs(x - y) <= limit for x, y in zip(a, b))

print(compatible((1, 2), (-1, 2), 1), compatible((1, 2), (1, 2), 1), compatible((1, 2), (2, 4), 1))   # False True False

False True False


### 3.3. LIS kiểu `dp[j] = 1 + max dp[i]`, và bitset số nguyên

**LIS** (dãy con tăng dài nhất) có DP: `dp[j] = 1 + max(dp[i])` với `i < j` và `a[i] < a[j]`. Ở bài này chỉ thay điều kiện "`a[i] < a[j]`" bằng "cột `i` và `j` tương thích".

Ngoài ra ta dùng **bitset** bằng số nguyên Python: bit `i` của một số cho biết "cột `i` thoả điều kiện". Phép `&` (và) giao hai tập; `(1 << j) - 1` là mặt nạ các cột `< j`. Duyệt các bit bật: `low = x & -x`, `i = low.bit_length() - 1`, `x ^= low`.

In [4]:
mask = 0b101101
print([i for i in range(6) if mask >> i & 1])          # cột 0, 2, 3, 5
before = mask & ((1 << 3) - 1)                          # chỉ giữ cột < 3
print(bin(before))
out = []
x = mask
while x:
    low = x & -x
    out.append(low.bit_length() - 1)
    x ^= low
print(out)

[0, 2, 3, 5]
0b101
[0, 2, 3, 5]


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2: các cột là `A=(1,2)`, `B=(-1,2)`, `C=(1,2)` với `limit = 1`.

- `A` và `B`: hàng 0 chênh `|1−(−1)| = 2 > 1` ⇒ **không** tương thích.
- `B` và `C`: tương tự, không.
- `A` và `C`: `(0, 0)` ⇒ tương thích.

Giữ `A, C` được 2 cột. Giữ cả `A, B, C` thì `A` và `B` liền kề ⇒ vi phạm. Đáp án 2.

Hai điểm cần chắc chắn:

1. **Chỉ cặp liền kề trong phần giữ lại phải tương thích.** Ví dụ `[[0, 2, 4]]`, `limit = 2`: giữ cả ba cột, các cặp liền kề chênh `2` và `2` — hợp lệ (dù cột 0 và 2 chênh `4`). Đáp án `3`.
2. Điều kiện phải đúng ở **mọi hàng** (AND qua các hàng), không phải một hàng.

Vậy: nếu coi mỗi cột là một đỉnh, cạnh `i → j` (`i < j`) tồn tại khi hai cột tương thích, thì đáp án là **số đỉnh của đường đi dài nhất** trong DAG này (các cạnh luôn đi từ chỉ số nhỏ tới lớn).

In [5]:
cols = list(zip(*[[1, -1, 1], [2, 2, 2]]))
for i in range(3):
    print([int(compatible(cols[i], cols[j], 1)) for j in range(3)])     # ma trận tương thích
print("ví dụ [[0,2,4]], limit=2: cột 0-1:", compatible((0,), (2,), 2), ", 1-2:", compatible((2,), (4,), 2), ", 0-2:", compatible((0,), (4,), 2))

[1, 0, 1]
[0, 1, 0]
[1, 0, 1]
ví dụ [[0,2,4]], limit=2: cột 0-1: True , 1-2: True , 0-2: False


## 5. Cách 1 — Brute Force: liệt kê mọi tập con cột

**Ý tưởng.** Với `n` cột có `2^n − 1` tập con khác rỗng. Với mỗi tập con (mặt nạ bit), lấy các chỉ số theo thứ tự tăng dần, kiểm tra từng cặp liền kề trên mọi hàng. Trả về kích thước lớn nhất của tập con hợp lệ.

Đây là bản dịch gần như nguyên văn của đề nên rất đáng tin để làm trọng tài.

In [6]:
def brute_force(grid, limit):
    m, n = len(grid), len(grid[0])
    best = 1
    for mask in range(1, 1 << n):
        cols = [c for c in range(n) if mask >> c & 1]
        if len(cols) <= best:
            continue
        ok = True
        for a, b in zip(cols, cols[1:]):                 # CHỈ cặp liền kề
            if any(abs(grid[r][b] - grid[r][a]) > limit for r in range(m)):
                ok = False
                break
        if ok:
            best = len(cols)
    return best

for grid, limit, expected in EXAMPLES:
    print(grid, limit, "->", brute_force(grid, limit), "(kỳ vọng", expected, ")")

[[-2, 0, 3]] 2 -> 2 (kỳ vọng 2 )
[[1, -1, 1], [2, 2, 2]] 1 -> 2 (kỳ vọng 2 )
[[-5, 5]] 9 -> 1 (kỳ vọng 1 )


**Độ phức tạp.** `O(2^n · n · m)`: chỉ dùng cho `n ≤ 12`. **Nút thắt:** ta kiểm tra lại cùng một cặp cột liền kề rất nhiều lần trong các tập con khác nhau. Chỉ cần quyết định **cột cuối cùng của dãy** và độ dài tốt nhất kết thúc ở đó.

## 6. Bước đột phá — DP kiểu LIS trên đồ thị tương thích

Định nghĩa: `dp[j]` = số cột nhiều nhất của một dãy nhất quán **kết thúc ở cột `j`**.

- `dp[j] = 1` (chỉ giữ cột `j`) hoặc
- `dp[j] = 1 + dp[i]` nếu `i < j` và cột `i`, `j` tương thích — vì dãy tốt nhất kết thúc ở `i`, thêm `j` vào cuối chỉ cần `(i, j)` tương thích (các cặp liền kề cũ đã hợp lệ).

Đáp án = `max(dp[j])`.

**Quy tắc "nếu … thì …":** nếu cặp `(i, j)` tương thích (mọi hàng chênh `≤ limit`) thì cho phép chuyển `dp[i] → dp[j]`.

**Vì sao chỉ cần `dp[i]` mà không cần biết dãy cụ thể?** Điều kiện của dãy chỉ ràng buộc **cột cuối** với cột sắp thêm; mọi thứ trước đó đã hợp lệ (tính chất "không có bộ nhớ" kiểu LIS).

## 7. Các cách tiếp theo

- **Cách 2:** DP `O(n² · m)` với `all` kiểm tra tương thích tại chỗ.
- **Cách 3:** dựng **ma trận tương thích** bằng `numpy`, rồi DP.
- **Cách 4:** dựng tương thích bằng **bitset + sắp xếp + `bisect`** theo từng hàng, rồi DP duyệt các bit — đáp án nộp.

### 7.1. Cách 2 — DP với kiểm tra tại chỗ

**Giải thích từng đoạn.**

1. `cols = list(zip(*grid))` lấy các cột (mỗi cột là tuple `m` số).
2. `dp = [1] * n`.
3. Với mỗi `j`, thử mọi `i < j`; nếu `compatible(cols[i], cols[j])` thì `dp[j] = max(dp[j], dp[i] + 1)`.
4. Đáp án `max(dp)`.

In [7]:
def dp_inplace(grid, limit):
    cols = list(zip(*grid))
    n = len(cols)
    dp = [1] * n
    for j in range(n):
        for i in range(j):
            if dp[i] + 1 > dp[j] and all(abs(x - y) <= limit for x, y in zip(cols[i], cols[j])):
                dp[j] = dp[i] + 1
    return max(dp)

for grid, limit, expected in EXAMPLES:
    assert dp_inplace(grid, limit) == expected
print("Cách 2 khớp 3 ví dụ")

Cách 2 khớp 3 ví dụ


**Độ phức tạp.** `O(n² · m)` trường hợp xấu: `250² / 2 · 250 ≈ 7.8·10^6` phép so sánh. Có tối ưu `dp[i] + 1 > dp[j]` để bỏ qua kiểm tra khi không thể cải thiện, nhưng trường hợp xấu (mọi cặp tương thích) vẫn nặng.

### 7.2. Cách 3 — Ma trận tương thích bằng `numpy`

**Ý tưởng.** Với mỗi cột `j`, tính độ lệch lớn nhất theo hàng so với **mọi** cột cùng lúc: `np.abs(G - G[:, [j]]).max(axis=0)` (broadcast cột `j` ra khắp các cột). So với `limit` được vector boolean `compat[j]`. Rồi DP chỉ việc tra bảng.

In [8]:
import numpy as np

def compat_numpy(grid, limit):
    G = np.array(grid, dtype=np.int64)                      # m x n
    n = G.shape[1]
    compat = np.zeros((n, n), dtype=bool)
    for j in range(n):
        diff = np.abs(G - G[:, [j]]).max(axis=0)            # độ lệch lớn nhất theo hàng giữa cột j và mọi cột
        compat[j] = diff <= limit
    return compat

def dp_numpy(grid, limit):
    compat = compat_numpy(grid, limit)
    n = compat.shape[0]
    dp = [1] * n
    for j in range(n):
        for i in np.nonzero(compat[j, :j])[0].tolist():
            if dp[i] + 1 > dp[j]:
                dp[j] = dp[i] + 1
    return max(dp)

for grid, limit, expected in EXAMPLES:
    assert dp_numpy(grid, limit) == expected
print("Cách 3 khớp 3 ví dụ")
print(compat_numpy([[1, -1, 1], [2, 2, 2]], 1).astype(int))

Cách 3 khớp 3 ví dụ
[[1 0 1]
 [0 1 0]
 [1 0 1]]


**Độ phức tạp.** Dựng bảng `O(n · m · n)` phép số học nhưng vector hoá bằng `numpy`; DP `O(n²)`. Bộ nhớ `O(n²)` cho bảng và `O(m·n)` cho tạm thời mỗi lần.

### 7.3. Cách 4 — Bitset theo hàng: sắp xếp + `bisect` (đáp án nộp)

**Ý tưởng.** Với **một hàng `r`**, tập cột tương thích với cột `j` là `{i : |g[r][i] − g[r][j]| ≤ limit}` — các cột có giá trị trong khoảng `[v − limit, v + limit]`. Nếu sắp xếp cột theo giá trị hàng `r`, đó là một **đoạn liên tục** trong thứ tự đã sắp. Ta cần bitset các **chỉ số cột** nằm trong đoạn đó.

**Mẹo tiền tố bitset.** Với thứ tự sắp xếp `order`, đặt `pre[k]` = OR của `1 << order[t]` với `t < k`. Vì các chỉ số cột khác nhau nên các bit không trùng; do đó bitset của đoạn `[lo, hi)` là `pre[hi] ^ pre[lo]` (hiệu tập). Dùng `bisect_left`, `bisect_right` tìm `lo`, `hi`.

Tương thích tổng = **AND** qua mọi hàng. Rồi DP: `dp[j] = 1 + max dp[i]` với `i` là các bit bật của `compat[j] & ((1 << j) - 1)`.

In [9]:
from bisect import bisect_left, bisect_right

def dp_bitset(grid, limit):
    m, n = len(grid), len(grid[0])
    full = (1 << n) - 1
    compat = [full] * n
    for r in range(m):
        row = grid[r]
        order = sorted(range(n), key=row.__getitem__)       # cột theo giá trị tăng dần
        vals = [row[c] for c in order]
        pre = [0] * (n + 1)
        for k, c in enumerate(order):
            pre[k + 1] = pre[k] | (1 << c)
        for j in range(n):
            v = row[j]
            lo = bisect_left(vals, v - limit)
            hi = bisect_right(vals, v + limit)
            compat[j] &= pre[hi] ^ pre[lo]
    dp = [1] * n
    for j in range(n):
        mask = compat[j] & ((1 << j) - 1)                    # chỉ các cột i < j tương thích
        best = 0
        while mask:
            low = mask & -mask
            i = low.bit_length() - 1
            mask ^= low
            if dp[i] > best:
                best = dp[i]
        dp[j] = best + 1
    return max(dp)

for grid, limit, expected in EXAMPLES:
    assert dp_bitset(grid, limit) == expected
print("Cách 4 khớp 3 ví dụ")

Cách 4 khớp 3 ví dụ


**Độ phức tạp.** Mỗi hàng: sắp xếp `O(n log n)`, dựng tiền tố `O(n)` phép OR (số nguyên `n` bit), mỗi cột `O(log n)` cho `bisect` và vài phép bit. Tổng dựng `O(m · n log n)` phép bit trên số `n` bit (≈ `n/64` từ máy mỗi phép). DP duyệt bit `O(n²)`. Bộ nhớ `O(n)` số nguyên `n` bit.

## 8. TRACE từng bước

Ta in ma trận tương thích và từng bước DP cho `grid = [[0,1,5,2,6,3],[0,0,4,1,5,2]]`, `limit = 2`.

| cột | ý nghĩa |
|---|---|
| `compat[j]` | các cột `i < j` tương thích với `j` |
| `dp[j]` | độ dài dãy nhất quán dài nhất kết thúc ở `j` |
| `từ` | cột `i` cho `dp[i] + 1` lớn nhất |

In [10]:
def trace_dp(grid, limit):
    cols = list(zip(*grid))
    n = len(cols)
    print("các cột:", cols)
    dp = [1] * n
    parent = [-1] * n
    for j in range(n):
        cand = [i for i in range(j) if compatible(cols[i], cols[j], limit)]
        for i in cand:
            if dp[i] + 1 > dp[j]:
                dp[j], parent[j] = dp[i] + 1, i
        print(f"j={j}: cột {cols[j]}  tương thích với {cand}  -> dp[{j}]={dp[j]}" + (f" (từ {parent[j]})" if parent[j] >= 0 else ""))
    end = max(range(n), key=lambda x: dp[x])
    chain = []
    while end != -1:
        chain.append(end); end = parent[end]
    print("dp =", dp, "-> đáp án", max(dp), "; dãy cột:", chain[::-1])
    return max(dp)

trace_dp([[0, 1, 5, 2, 6, 3], [0, 0, 4, 1, 5, 2]], 2)

các cột: [(0, 0), (1, 0), (5, 4), (2, 1), (6, 5), (3, 2)]
j=0: cột (0, 0)  tương thích với []  -> dp[0]=1
j=1: cột (1, 0)  tương thích với [0]  -> dp[1]=2 (từ 0)
j=2: cột (5, 4)  tương thích với []  -> dp[2]=1
j=3: cột (2, 1)  tương thích với [0, 1]  -> dp[3]=3 (từ 1)
j=4: cột (6, 5)  tương thích với [2]  -> dp[4]=2 (từ 2)
j=5: cột (3, 2)  tương thích với [1, 2, 3]  -> dp[5]=4 (từ 3)
dp = [1, 2, 1, 3, 2, 4] -> đáp án 4 ; dãy cột: [0, 1, 3, 5]


4

Đọc trace: cột `1` và `2` có hàng 0 chênh `|1−5| = 4 > 2` nên không tương thích. Dãy dài nhất đi qua các cột có giá trị gần nhau ở cả hai hàng. Trace thứ hai cho ví dụ 1 của đề (một hàng, `limit = 2`):

In [11]:
trace_dp([[-2, 0, 3]], 2)

các cột: [(-2,), (0,), (3,)]
j=0: cột (-2,)  tương thích với []  -> dp[0]=1
j=1: cột (0,)  tương thích với [0]  -> dp[1]=2 (từ 0)
j=2: cột (3,)  tương thích với []  -> dp[2]=1
dp = [1, 2, 1] -> đáp án 2 ; dãy cột: [0, 1]


2

## 9. Code nộp LeetCode

In [12]:
from typing import List
from bisect import bisect_left, bisect_right

class Solution:
    def maxConsistentColumns(self, grid: List[List[int]], limit: int) -> int:
        m, n = len(grid), len(grid[0])
        compat = [(1 << n) - 1] * n                 # compat[j]: bitset các cột tương thích với cột j
        for r in range(m):
            row = grid[r]
            order = sorted(range(n), key=row.__getitem__)
            vals = [row[c] for c in order]
            pre = [0] * (n + 1)
            for k, c in enumerate(order):
                pre[k + 1] = pre[k] | (1 << c)      # tiền tố bitset theo thứ tự giá trị
            for j in range(n):
                lo = bisect_left(vals, row[j] - limit)
                hi = bisect_right(vals, row[j] + limit)
                compat[j] &= pre[hi] ^ pre[lo]      # cột có giá trị trong [v-limit, v+limit]

        dp = [1] * n                                # dp[j]: dãy nhất quán dài nhất kết thúc ở cột j
        for j in range(n):
            mask = compat[j] & ((1 << j) - 1)       # chỉ xét i < j
            best = 0
            while mask:
                low = mask & -mask
                i = low.bit_length() - 1
                mask ^= low
                if dp[i] > best:
                    best = dp[i]
            dp[j] = best + 1
        return max(dp)

for grid, limit, expected in EXAMPLES:
    assert Solution().maxConsistentColumns(grid, limit) == expected
print("Code nộp khớp 3 ví dụ")

Code nộp khớp 3 ví dụ


## 10. Vì sao thuật toán đúng

**(a) Cấu trúc bài toán.** Một dãy cột `c_1 < ... < c_k` là hợp lệ ⇔ mọi cặp liền kề `(c_t, c_{t+1})` tương thích. Điều kiện chỉ ràng buộc các cặp liền kề, và cạnh `i → j` chỉ có khi `i < j`, nên bài toán là **đường đi dài nhất trong DAG** (mỗi dãy hợp lệ là một đường đi và ngược lại).

**(b) DP.** `dp[j]` = số đỉnh của đường đi dài nhất kết thúc ở `j`. Một đường đi kết thúc ở `j` hoặc chỉ gồm `j`, hoặc là một đường đi kết thúc ở `i` cộng cạnh `i → j`. Vì phần đường đi trước `j` độc lập với những gì sau, tối ưu con đúng: `dp[j] = 1 + max{dp[i] : i<j, compat(i,j)}`. Đáp án `max dp[j]`.

**(c) Bitset tương thích đúng.** Với một hàng, `|g[i] − g[j]| ≤ limit ⇔ g[i] ∈ [g[j] − limit, g[j] + limit]`. Sau khi sắp xếp theo giá trị, tập các cột có giá trị trong khoảng này là đoạn `[lo, hi)` với `lo = bisect_left(vals, v − limit)` và `hi = bisect_right(vals, v + limit)`. `pre[hi] ^ pre[lo]` là bitset đúng các cột trong đoạn vì `pre[lo] ⊆ pre[hi]` và các bit rời nhau.

**(d) AND qua các hàng.** Cột `i` tương thích với `j` ⇔ mọi hàng thoả điều kiện ⇔ `i` thuộc giao của các bitset theo hàng ⇒ `compat[j] &= ...` cho mỗi hàng.

**(e) Biên.** `limit = 0`: khoảng `[v, v]`, chỉ các cột bằng nhau ở mọi hàng. `n = 1`: không có `i < j`, `dp = [1]`. Đáp án luôn `≥ 1` vì `dp[j] ≥ 1`.

## 11. Phân tích Time/Space

Gọi `m` = số hàng, `n` = số cột.

| Cách | Time | Space |
|---|---|---|
| Brute force | `O(2^n · n · m)` | `O(n)` |
| DP kiểm tra tại chỗ | `O(n² · m)` | `O(n)` |
| Bảng tương thích numpy + DP | `O(n² · m)` (vector hoá) + `O(n²)` | `O(n²)` |
| Bitset + DP (nộp) | `O(m · n · (log n + n/w)) + O(n²)`, `w = 64` | `O(n)` số nguyên `n` bit |

Đo trên trường hợp xấu `m = n = 250` (giá trị gần nhau, mọi cặp tương thích):

In [13]:
import time, random

def timed(fn, grid, limit):
    t0 = time.time()
    r = fn(grid, limit)
    return r, round(time.time() - t0, 3)

rng = random.Random(1)
m = n = 250
grid_all = [[rng.randint(-100, 100) for _ in range(n)] for _ in range(m)]
print("mọi cặp tương thích (limit=10^5):")
for name, fn in [("tại chỗ", dp_inplace), ("numpy", dp_numpy), ("bitset", dp_bitset), ("nộp", Solution().maxConsistentColumns)]:
    print(f"  {name:8s}", timed(fn, grid_all, 10**5))
grid_mid = [[rng.randint(-10**5, 10**5) for _ in range(n)] for _ in range(m)]
print("giá trị ngẫu nhiên lớn, limit = 150000 (hầu như không cặp nào tương thích ở cả 250 hàng):")
for name, fn in [("tại chỗ", dp_inplace), ("numpy", dp_numpy), ("bitset", dp_bitset), ("nộp", Solution().maxConsistentColumns)]:
    print(f"  {name:8s}", timed(fn, grid_mid, 150000))
grid_few = [[rng.randint(0, 100) for _ in range(n)] for _ in range(3)]
print("3 hàng, 250 cột, giá trị 0..100, limit = 50 (đồ thị tương thích dày vừa phải):")
for name, fn in [("tại chỗ", dp_inplace), ("numpy", dp_numpy), ("bitset", dp_bitset), ("nộp", Solution().maxConsistentColumns)]:
    print(f"  {name:8s}", timed(fn, grid_few, 50))
grid_none = [[(j * 997 + r) for j in range(n)] for r in range(m)]
print("không cặp nào tương thích (limit=0):")
for name, fn in [("tại chỗ", dp_inplace), ("bitset", dp_bitset)]:
    print(f"  {name:8s}", timed(fn, grid_none, 0))

mọi cặp tương thích (limit=10^5):


  tại chỗ  (250, 0.473)
  numpy    (250, 0.021)
  bitset   (250, 0.025)
  nộp      (250, 0.025)
giá trị ngẫu nhiên lớn, limit = 150000 (hầu như không cặp nào tương thích ở cả 250 hàng):
  tại chỗ  (1, 0.037)
  numpy    (1, 0.019)
  bitset   (1, 0.023)
  nộp      (1, 0.023)
3 hàng, 250 cột, giá trị 0..100, limit = 50 (đồ thị tương thích dày vừa phải):
  tại chỗ  (126, 0.008)
  numpy    (126, 0.002)
  bitset   (126, 0.002)
  nộp      (126, 0.002)
không cặp nào tương thích (limit=0):


  tại chỗ  (1, 0.01)
  bitset   (1, 0.02)


## 12. Bộ test

Ví dụ đề, edge case, phản ví dụ cho các cách nghĩ sai, đối chiếu ngẫu nhiên: vét cạn ↔ DP tại chỗ ↔ numpy ↔ bitset ↔ code nộp.

In [14]:
S = Solution()
for grid, limit, expected in EXAMPLES:
    for fn in (brute_force, dp_inplace, dp_numpy, dp_bitset, S.maxConsistentColumns):
        assert fn(grid, limit) == expected, (fn.__name__, grid)
print("Ví dụ đề: mọi cách khớp")

Ví dụ đề: mọi cách khớp


In [15]:
# Edge case
assert S.maxConsistentColumns([[5]], 0) == 1                          # một cột
assert S.maxConsistentColumns([[1, 1, 1, 1]], 0) == 4                 # toàn bằng nhau
assert S.maxConsistentColumns([[0, 1, 2, 3]], 0) == 1                 # limit = 0, không cột nào bằng nhau
assert S.maxConsistentColumns([[0, 2, 4]], 2) == 3                    # chỉ cặp liền kề cần chênh <= 2 (0 và 4 chênh 4)
assert S.maxConsistentColumns([[0, 5, 0, 5]], 0) == 2                 # giữ hai cột 0, hoặc hai cột 5
assert S.maxConsistentColumns([[0, 100, 1, 101, 2]], 2) == 3          # 0,1,2
assert S.maxConsistentColumns([[1], [2], [3]], 5) == 1                # một cột, nhiều hàng
assert S.maxConsistentColumns([[0, 0], [0, 5]], 4) == 1               # hàng 1 vi phạm
assert S.maxConsistentColumns([[-10**5, 10**5]], 2 * 10**5) == 2      # biên: chênh đúng 2e5
assert S.maxConsistentColumns([[-10**5, 10**5]], 2 * 10**5 - 1) == 1
print("Edge case: OK")

Edge case: OK


**Phản ví dụ** cho các cách nghĩ sai (xem thêm mục 13): kiểm tra mọi cặp (không chỉ liền kề), tham lam "giữ nếu tương thích với cột vừa giữ", dùng `<` thay `<=`, chỉ xét một hàng.

In [16]:
# 1) Đối chiếu ngẫu nhiên nhỏ với vét cạn
rng = random.Random(2026)
for it in range(500):
    m = rng.randint(1, 3)
    n = rng.randint(1, 10)
    hi = rng.choice([2, 5, 20])
    grid = [[rng.randint(-hi, hi) for _ in range(n)] for _ in range(m)]
    limit = rng.randint(0, hi)
    ref = brute_force(grid, limit)
    assert dp_inplace(grid, limit) == ref, (grid, limit)
    assert dp_numpy(grid, limit) == ref, (grid, limit)
    assert dp_bitset(grid, limit) == ref, (grid, limit)
    assert S.maxConsistentColumns(grid, limit) == ref, (grid, limit)
print("500 input ngẫu nhiên nhỏ: khớp vét cạn")

# 2) Input trung bình: DP so với nhau
rng = random.Random(77)
for it in range(60):
    m = rng.randint(1, 8)
    n = rng.randint(10, 60)
    hi = rng.choice([5, 30, 1000])
    grid = [[rng.randint(-hi, hi) for _ in range(n)] for _ in range(m)]
    limit = rng.randint(0, hi)
    a = S.maxConsistentColumns(grid, limit)
    assert a == dp_inplace(grid, limit) == dp_numpy(grid, limit) == dp_bitset(grid, limit)
print("60 input trung bình: các DP khớp nhau")

# 3) Input lớn: bitset ↔ numpy
for it in range(5):
    m, n = rng.randint(100, 250), rng.randint(100, 250)
    grid = [[rng.randint(-10**5, 10**5) for _ in range(n)] for _ in range(m)]
    limit = rng.randint(10**4, 10**5)
    assert S.maxConsistentColumns(grid, limit) == dp_numpy(grid, limit)
print("5 input lớn: bitset ↔ numpy khớp")

500 input ngẫu nhiên nhỏ: khớp vét cạn
60 input trung bình: các DP khớp nhau


5 input lớn: bitset ↔ numpy khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Kiểm tra **mọi cặp** cột được giữ (quá chặt)

Đề chỉ ràng buộc cặp **liền kề** trong phần giữ lại. Nếu bắt mọi cặp tương thích ta sẽ bỏ lỡ lời giải.

In [17]:
def wrong_all_pairs(grid, limit):
    m, n = len(grid), len(grid[0])
    best = 1
    for mask in range(1, 1 << n):
        cols = [c for c in range(n) if mask >> c & 1]
        if all(abs(grid[r][a] - grid[r][b]) <= limit for r in range(m) for a in cols for b in cols):   # SAI
            best = max(best, len(cols))
    return best

g = [[0, 2, 4]]
print("sai :", wrong_all_pairs(g, 2))                      # 2
print("đúng:", Solution().maxConsistentColumns(g, 2))      # 3

sai : 2
đúng: 3


**Cách đúng:** chỉ so `cols[t]` với `cols[t+1]`.

### Lỗi 2 — Tham lam: duyệt trái sang phải, giữ cột nếu tương thích với cột vừa giữ

Nhìn hợp lý nhưng có thể "ôm" một cột xấu làm mất lời giải tốt hơn. Ta tìm phản ví dụ bằng tìm kiếm ngẫu nhiên có seed.

In [18]:
def greedy_last(grid, limit):
    cols = list(zip(*grid))
    kept = [cols[0]]
    for c in cols[1:]:
        if compatible(kept[-1], c, limit):
            kept.append(c)
    return len(kept)

rng = random.Random(3)
found = None
for _ in range(20000):
    m, n = rng.randint(1, 2), rng.randint(3, 7)
    grid = [[rng.randint(0, 6) for _ in range(n)] for _ in range(m)]
    limit = rng.randint(0, 3)
    if greedy_last(grid, limit) != brute_force(grid, limit):
        found = (grid, limit); break
assert found is not None
print("grid =", found[0], "limit =", found[1])
print("tham lam:", greedy_last(*found), "| đúng:", brute_force(*found))

grid = [[6, 0, 3, 1, 0]] limit = 2
tham lam: 1 | đúng: 3


**Vì sao sai:** giữ cột đầu tiên và chỉ giữ tiếp khi tương thích với cột vừa giữ có thể kẹt ở một "nhánh" ngắn, trong khi bỏ cột đầu rồi bắt đầu ở nhánh khác cho dãy dài hơn. **Cách đúng:** DP thử mọi tiền nhiệm `i`.

### Lỗi 3 — Dùng `<` thay vì `<=`

Đề cho phép chênh **đúng bằng** `limit`.

In [19]:
def wrong_strict(grid, limit):
    cols = list(zip(*grid))
    n = len(cols)
    dp = [1] * n
    for j in range(n):
        for i in range(j):
            if all(abs(x - y) < limit for x, y in zip(cols[i], cols[j])):      # SAI: < thay vì <=
                dp[j] = max(dp[j], dp[i] + 1)
    return max(dp)

print("sai :", wrong_strict([[-2, 0, 3]], 2))                 # 1
print("đúng:", Solution().maxConsistentColumns([[-2, 0, 3]], 2))   # 2

sai : 1
đúng: 2


**Cách đúng:** `abs(a - b) <= limit`.

### Lỗi 4 — Chỉ xét một hàng (hoặc dùng OR thay AND giữa các hàng)

Tương thích phải đúng ở **mọi** hàng.

In [20]:
def wrong_any_row(grid, limit):
    cols = list(zip(*grid))
    n = len(cols)
    dp = [1] * n
    for j in range(n):
        for i in range(j):
            if any(abs(x - y) <= limit for x, y in zip(cols[i], cols[j])):     # SAI: any thay vì all
                dp[j] = max(dp[j], dp[i] + 1)
    return max(dp)

g = [[1, -1, 1], [2, 2, 2]]
print("sai :", wrong_any_row(g, 1))                            # 3: hàng 1 toàn bằng nhau nên 'any' đúng ở mọi cặp
print("đúng:", Solution().maxConsistentColumns(g, 1))          # 2

sai : 3
đúng: 2


**Cách đúng:** `all` qua các hàng (với bitset: `AND` các bitset hàng).

## 14. Mẹo & nhận dạng pattern

- **"Xoá phần tử để dãy còn lại thoả điều kiện giữa hai phần tử liền kề"** → DP kiểu LIS: `dp[j] = 1 + max dp[i]` với `i < j` thoả quan hệ.
- Phân biệt **điều kiện liền kề** (DP) với **điều kiện mọi cặp** (clique/khác hẳn): đọc kỹ đề.
- **Nhiều chiều (nhiều hàng)** → quan hệ giữa hai cột là **AND** các điều kiện từng hàng.
- **Điều kiện kiểu `|a − b| ≤ limit` trên một hàng** → khoảng giá trị liên tục → sắp xếp + `bisect` + bitset tiền tố.
- **Bitset bằng số nguyên Python**: `&`, `>>`, `x & -x` rất nhanh với `n ≤ 250`; `pre[hi] ^ pre[lo]` lấy bitset của một đoạn khi các bit rời nhau.
- Với `n ≤ 250`, `O(n²)` thoải mái; chú ý hằng số `m` nhân vào.
- Luôn thử phản ví dụ cho tham lam trước khi tin.

## 15. Biến thể

1. **Khôi phục dãy cột** đạt tối đa (in ra các chỉ số).
2. **Đếm số tập con nhất quán** (khác rỗng).
3. **Chỉ được xoá các cột liên tiếp?** (giữ một đoạn liên tục): cửa sổ trượt.
4. **Điều kiện theo cột thay vì hàng**, hoặc trọng số mỗi cột (tổng trọng số lớn nhất): DP với trọng số.

Ta cài thử biến thể 1 và 2 (đếm), kiểm chứng bằng vét cạn.

In [21]:
def best_columns(grid, limit):
    cols = list(zip(*grid))
    n = len(cols)
    dp = [1] * n
    par = [-1] * n
    for j in range(n):
        for i in range(j):
            if dp[i] + 1 > dp[j] and compatible(cols[i], cols[j], limit):
                dp[j], par[j] = dp[i] + 1, i
    end = max(range(n), key=lambda x: dp[x])
    chain = []
    while end != -1:
        chain.append(end); end = par[end]
    return chain[::-1]

def count_consistent_subsets(grid, limit):
    cols = list(zip(*grid))
    n = len(cols)
    cnt = [1] * n                                    # cnt[j] = số dãy hợp lệ kết thúc ở j (kể cả chỉ {j})
    for j in range(n):
        for i in range(j):
            if compatible(cols[i], cols[j], limit):
                cnt[j] += cnt[i]
    return sum(cnt)

def count_bruteforce(grid, limit):
    m, n = len(grid), len(grid[0])
    total = 0
    for mask in range(1, 1 << n):
        cols = [c for c in range(n) if mask >> c & 1]
        if all(abs(grid[r][b] - grid[r][a]) <= limit for a, b in zip(cols, cols[1:]) for r in range(m)):
            total += 1
    return total

print(best_columns([[0, 1, 5, 2, 6, 3], [0, 0, 4, 1, 5, 2]], 2))
rng = random.Random(15)
for _ in range(200):
    m, n = rng.randint(1, 3), rng.randint(1, 9)
    grid = [[rng.randint(-4, 4) for _ in range(n)] for _ in range(m)]
    limit = rng.randint(0, 4)
    ch = best_columns(grid, limit)
    assert len(ch) == S.maxConsistentColumns(grid, limit)
    assert all(all(abs(grid[r][b] - grid[r][a]) <= limit for r in range(m)) for a, b in zip(ch, ch[1:]))
    assert count_consistent_subsets(grid, limit) == count_bruteforce(grid, limit)
print("Biến thể khôi phục dãy và đếm tập con: khớp vét cạn")

[0, 1, 3, 5]
Biến thể khôi phục dãy và đếm tập con: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn tập con | thử mọi tập con cột | `O(2^n·n·m)` | `O(n)` | trọng tài, `n ≤ 12` |
| 2. DP tại chỗ | `dp[j]=1+max dp[i]`, `all` mỗi cặp | `O(n²·m)` | `O(n)` | dễ viết, `n·m` vừa |
| 3. numpy + DP | bảng tương thích vector hoá | `O(n²·m)` + `O(n²)` | `O(n²)` | muốn tránh vòng Python, có numpy |
| 4. Bitset (nộp) | sắp xếp + `bisect` + prefix bitset | `O(m·n log n + n²)` | `O(n)` | ổn định, nhanh, không phụ thuộc thư viện |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `compatible((1, 2), (2, 4), 1)` và `compatible((1, 2), (2, 3), 1)` là gì?

**Bài 2 (tính tay).** `grid = [[1, 2, 3, 9, 10]]`, `limit = 1`. Đáp án là bao nhiêu và giữ cột nào?

**Bài 3 (điền TODO).** Hoàn thiện `compat_matrix(grid, limit)` trả về ma trận `n×n` boolean (có thể dùng vòng lặp thường).

**Bài 4 (tự viết lại).** Viết `my_solve(grid, limit)` dùng `compat_matrix` rồi DP kiểu LIS.

**Bài 5 (biến thể / mini challenge).** Viết `max_weight(grid, limit, w)` trong đó `w[j]` là trọng số (không âm) của cột `j`; tìm tổng trọng số lớn nhất của một dãy cột nhất quán. Kiểm chứng bằng vét cạn.

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def compat_matrix(grid, limit):
    cols = list(zip(*grid))
    n = len(cols)
    # TODO: trả về M với M[i][j] = True nếu mọi hàng thoả |cols[i][r] - cols[j][r]| <= limit
    pass

# Bài 4 — TODO
def my_solve(grid, limit):
    # TODO: dp = [1] * n; dp[j] = max(dp[j], dp[i] + 1) nếu i < j và M[i][j]; trả max(dp)
    pass

# Bài 5 — TODO
def max_weight(grid, limit, w):
    # TODO: dp[j] = w[j] + max(0, max dp[i] với i < j tương thích); trả max(dp)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: so từng hàng với `limit = 1`.
- Mức 2: `(1,2)` vs `(2,4)`: hàng 1 chênh `2`; `(1,2)` vs `(2,3)`: chênh `1` và `1`.
- Mức 3: `False` và `True`.

**Bài 2.**
- Mức 1: cột nào chênh `≤ 1` với cột kế?
- Mức 2: `1,2,3` liền nhau, `9,10` liền nhau; `3` và `9` chênh 6.
- Mức 3: đáp án `3` (cột 0, 1, 2).

**Bài 3.**
- Mức 1: hai vòng for trên `n`.
- Mức 2: `all(abs(a - b) <= limit for a, b in zip(cols[i], cols[j]))`.
- Mức 3: `[[compat(i, j) for j in range(n)] for i in range(n)]`.

**Bài 4.**
- Mức 1: `dp = [1] * n`.
- Mức 2: duyệt `j`, rồi `i < j`.
- Mức 3: `return max(dp)`.

**Bài 5.**
- Mức 1: thay "1" bằng `w[j]`.
- Mức 2: khởi tạo `dp[j] = w[j]`.
- Mức 3: `dp[j] = max(dp[j], dp[i] + w[j])` khi `i < j` tương thích.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
assert compatible((1, 2), (2, 4), 1) is False and compatible((1, 2), (2, 3), 1) is True

# Bài 2
assert Solution().maxConsistentColumns([[1, 2, 3, 9, 10]], 1) == 3
assert best_columns([[1, 2, 3, 9, 10]], 1) == [0, 1, 2]

# Bài 3
def compat_matrix_ans(grid, limit):
    cols = list(zip(*grid))
    n = len(cols)
    return [[all(abs(a - b) <= limit for a, b in zip(cols[i], cols[j])) for j in range(n)] for i in range(n)]

# Bài 4
def my_solve_ans(grid, limit):
    Mx = compat_matrix_ans(grid, limit)
    n = len(Mx)
    dp = [1] * n
    for j in range(n):
        for i in range(j):
            if Mx[i][j] and dp[i] + 1 > dp[j]:
                dp[j] = dp[i] + 1
    return max(dp)

rng = random.Random(33)
for _ in range(300):
    m, n = rng.randint(1, 3), rng.randint(1, 10)
    grid = [[rng.randint(-5, 5) for _ in range(n)] for _ in range(m)]
    limit = rng.randint(0, 5)
    assert my_solve_ans(grid, limit) == Solution().maxConsistentColumns(grid, limit)
print("Bài 1–4 OK")

Bài 1–4 OK


In [24]:
# Bài 5
def max_weight_ans(grid, limit, w):
    Mx = compat_matrix_ans(grid, limit)
    n = len(Mx)
    dp = list(w)
    for j in range(n):
        for i in range(j):
            if Mx[i][j] and dp[i] + w[j] > dp[j]:
                dp[j] = dp[i] + w[j]
    return max(dp)

def max_weight_brute(grid, limit, w):
    m, n = len(grid), len(grid[0])
    best = 0
    for mask in range(1, 1 << n):
        cols = [c for c in range(n) if mask >> c & 1]
        if all(abs(grid[r][b] - grid[r][a]) <= limit for a, b in zip(cols, cols[1:]) for r in range(m)):
            best = max(best, sum(w[c] for c in cols))
    return best

assert max_weight_ans([[1, 2, 3, 9, 10]], 1, [1, 1, 1, 5, 5]) == 10
assert max_weight_ans([[1, 2, 3, 9, 10]], 1, [1, 1, 1, 1, 1]) == 3
rng = random.Random(44)
for _ in range(200):
    m, n = rng.randint(1, 3), rng.randint(1, 9)
    grid = [[rng.randint(-4, 4) for _ in range(n)] for _ in range(m)]
    limit = rng.randint(0, 4)
    w = [rng.randint(0, 6) for _ in range(n)]
    assert max_weight_ans(grid, limit, w) == max_weight_brute(grid, limit, w), (grid, limit, w)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "LIS trên đồ thị tương thích + bitset":**

```python
compat = [(1 << n) - 1] * n
for r in range(m):                                   # mỗi hàng: bitset cột trong [v - limit, v + limit]
    order = sorted(range(n), key=grid[r].__getitem__)
    vals = [grid[r][c] for c in order]
    pre = [0] * (n + 1)
    for k, c in enumerate(order):
        pre[k + 1] = pre[k] | (1 << c)
    for j in range(n):
        lo = bisect_left(vals, grid[r][j] - limit)
        hi = bisect_right(vals, grid[r][j] + limit)
        compat[j] &= pre[hi] ^ pre[lo]               # AND qua các hàng
dp = [1] * n
for j in range(n):
    mask = compat[j] & ((1 << j) - 1)                # i < j
    best = 0
    while mask:
        low = mask & -mask; i = low.bit_length() - 1; mask ^= low
        best = max(best, dp[i])
    dp[j] = best + 1
return max(dp)
```

**Checklist trước khi submit:**

- [ ] Điều kiện chỉ áp dụng cho cặp **liền kề** của dãy được giữ.
- [ ] Tương thích = `AND` qua **mọi hàng**; dùng `<=` (không phải `<`).
- [ ] `dp[j]` khởi tạo `1`; đáp án `max(dp)` (luôn `≥ 1`).
- [ ] Bitset đoạn: `pre[hi] ^ pre[lo]` (các bit rời nhau); chỉ xét `i < j`.
- [ ] Đã đối chiếu vét cạn trên input nhỏ; đã thử phản ví dụ cho tham lam.